In [1]:
import pandas as pd 
import numpy as np 
from bs4 import BeautifulSoup
import requests
from io import StringIO
import re

ModuleNotFoundError: No module named 'requests'

In [3]:
url1 = "https://en.wikipedia.org/wiki/List_of_musicals_adapted_into_feature_films"

In [4]:
headers = {"User-Agent": "MyWikipedeaScraperForACollegeClass_OnHPLaptop15UsingExplorer (jessieaolsen@gmail.com)"}
r = requests.get(url1, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")
r.status_code

200

In [5]:
tables = pd.read_html(StringIO(r.text))
adaptations = {}
for table in tables:
    if {'Musical', 'Film'}.issubset(table.columns):
        table['Musical'] = table['Musical'].ffill()

        for _, row in table.dropna(subset=['Film']).iterrows():
            musical = row['Musical']
            if musical == 'Wicked (2003)':
                adaptations.setdefault(musical, []).append('Wicked (2024)')
                adaptations.setdefault(musical, []).append('Wicked: For Good (2025)')

            else:
                film = row['Film']
                adaptations.setdefault(musical, []).append(film)

            

print(adaptations)

{'13 (2007)': ['13: The Musical (2022)'], '1776 (1969)': ['1776 (1972)'], 'Alma, Where Do You Live? (1910)': ['Alma, Where Do You Live? (1917)'], 'Animal Crackers (1928)': ['Animal Crackers (1930)'], 'Annie (1976)': ['Annie (1982)', 'Annie: A Royal Adventure! (1995)', 'Annie (1999)', 'Annie (2014)'], 'Annie Get Your Gun (1946)': ['Annie Get Your Gun (1950)', 'Annie Get Your Gun (1957)', 'Annie Get Your Gun (1967)'], 'Anything Goes (1934)': ['Anything Goes (1936)', 'Anything Goes (1956)'], 'Are You with It? (1945)': ['Are You with It? (1939)'], 'Av. Larco: El Musical': ['Av. Larco (2017)'], 'Babes in Arms (1937)': ['Babes in Arms (1939)'], 'Balalaika (1936)': ['Balalaika (1939)'], 'The Band Wagon (1931)': ['Dancing in the Dark (1949)', 'The Band Wagon (1953)'], 'Beatlemania (1977)': ['Beatlemania: The Movie (1981)'], 'Been So Long (2009)': ['Been So Long (2018)'], 'Bells Are Ringing (1956)': ['Bells Are Ringing (1960)'], 'Best Foot Forward (1941)': ['Best Foot Forward (1943)'], 'The Bes

In [6]:
len(adaptations)

188

In [25]:
import os
from dotenv import load_dotenv
load_dotenv()

apikey = os.getenv("MY_API_KEY")

rt = {}
imdb = {}
for musical in adaptations:
    for movie in adaptations[musical]:
        title = re.match(r'^(.*?)\s*(\(\d{4}\))$', movie).group(1)
        year = re.match(r'^(.*?)\s*\((\d{4})\)$', movie).group(2)
        print(title, year)
        url = f"http://www.omdbapi.com/?t={title}&y={year}&apikey={apikey}"
        response = requests.get(url).json()
        imdbrating = response.get('imdbRating') if response.get('imdbRating') != 'N/A' else None
        rtrating = None
        for rating in response.get("Ratings", []):
            if rating.get("Source") == "Rotten Tomatoes":
                rtrating = rating.get("Value")
                break
        rt[movie] = rtrating
        imdb[movie] = imdbrating    


13: The Musical 2022
1776 1972
Alma, Where Do You Live? 1917
Animal Crackers 1930
Annie 1982
Annie: A Royal Adventure! 1995
Annie 1999
Annie 2014
Annie Get Your Gun 1950
Annie Get Your Gun 1957
Annie Get Your Gun 1967
Anything Goes 1936
Anything Goes 1956
Are You with It? 1939
Av. Larco 2017
Babes in Arms 1939
Balalaika 1939
Dancing in the Dark 1949
The Band Wagon 1953
Beatlemania: The Movie 1981
Been So Long 2018
Bells Are Ringing 1960
Best Foot Forward 1943
The Best Little Whorehouse in Texas 1982
The Romance of Old Bill 1918
The Better 'Ole 1926
Big Band 1930
Black Nativity 2013
The Boy Friend 1971
The Boys from Syracuse 1940
Bran Nue Dae 2009
Brigadoon 1954
Brigadoon 1966
Bye Bye Birdie 1963
Bye Bye Birdie 1995
Cabaret 1972
Cabin in the Sky 1943
Camelot 1967
Can-Can 1960
Carmen Jones 1954
Carousel 1956
Carousel 1967
The Cat and the Fiddle 1934
Chicago 2002
A Chorus Line 1985
Dark Streets 2008
The Cocoanuts 1929
Cyrano 2021
Damn Yankees 1958
Damn Yankees! 1967
The Deputy Drummer 193

In [8]:
url2 = "https://en.wikipedia.org/wiki/Tony_Award_for_Best_Musical"

In [9]:
headers = {"User-Agent": "MyWikipedeaScraperForACollegeClass_OnHPLaptop15UsingExplorer (jessieaolsen@gmail.com)"}
r2 = requests.get(url2, headers=headers)
soup2 = BeautifulSoup(r2.text, "html.parser")
r2.status_code

200

In [10]:
tables = pd.read_html(StringIO(r2.text))
tonys = []
for table in tables:
    if 'Musical' not in table.columns:
        continue
    table['Musical'] = table['Musical'].ffill()
    for _, row in table.iterrows():
        if pd.notna(row['Musical']):
            match = re.search(r'(.*?)\s*(?:–|-)?\s*(?:Book by|Conceived by|Book & Lyrics)', row['Musical'])
            if match:
                name = match.group(1).strip()
                if name not in tonys:
                    tonys.append(name)


In [11]:
print(tonys)

['Kiss Me, Kate', 'South Pacific', 'Guys and Dolls', 'The King and I', 'Wonderful Town', 'Kismet', 'The Pajama Game', 'Damn Yankees', 'Pipe Dream', 'My Fair Lady', 'Bells Are Ringing', 'Candide', 'Jamaica', 'New Girl in Town', 'Oh, Captain!', 'West Side Story', 'Redhead', 'Flower Drum Song', 'La Plume de Ma Tante', 'The Sound of Music', 'Fiorello!', 'Gypsy', 'Once Upon a Mattress', 'Take Me Along', 'Bye Bye Birdie', 'Do Re Mi', 'Irma La Douce', 'How to Succeed in Business Without Really Trying', 'Carnival!', 'Milk and Honey', 'No Strings', 'A Funny Thing Happened on the Way to the Forum', 'Little Me', 'Hello, Dolly!', 'Funny Girl', 'She Loves Me', 'Fiddler on the Roof', 'Golden Boy', 'Half a Sixpence', 'Oh, What a Lovely War!', 'Man of La Mancha', 'Mame', 'Skyscraper', 'Sweet Charity', 'Cabaret', 'I Do! I Do!', 'The Apple Tree', 'Walking Happy', 'Hallelujah, Baby!', 'The Happy Time', 'How Now, Dow Jones', 'Illya Darling', '1776', 'Hair', 'Promises, Promises', 'Zorba', 'Applause', 'Coco

In [ ]:
rows = []

for musical, movies in adaptations.items():
    for movie in movies:
        rows.append({"Musical": musical, "Movie": movie})

df = pd.DataFrame(rows)

df["IMDb"] = df["Movie"].map(imdb)
df["RottenTomatoes"] = df["Movie"].map(rt)

df["TonyNominatedMusical"] = df["Musical"].str.extract(r'(.*?)\s\(\d{4}\)')[0].isin(tonys)

df.head(10)

,Musical,Movie,IMDb,RottenTomatoes,TonyNominatedMusical
0,13 (2007),13: The Musical (2022),5.2,58%,False
1,1776 (1969),1776 (1972),7.6,68%,True
2,"Alma, Where Do You Live? (1910)","Alma, Where Do You Live? (1917)",NaN,NaN,False
3,Animal Crackers (1928),Animal Crackers (1930),7.4,97%,False
4,Annie (1976),Annie (1982),6.6,49%,True
5,Annie (1976),Annie: A Royal Adventure! (1995),5.0,NaN,True
6,Annie (1976),Annie (1999),6.7,NaN,True
7,Annie (1976),Annie (2014),5.4,28%,True
8,Annie Get Your Gun (1946),Annie Get Your Gun (1950),6.8,100%,False
9,Annie Get Your Gun (1946),Annie Get Your Gun (1957),7.5,NaN,False


In [23]:
df.to_csv("musical_adaptations.csv", index=False)